# tp-05l ランキング学習（lambdarank / rank_xendcg）のスコアを、確率に直してアンサンブルに加える

tp-05a では、ランキング系のスコアを「レース内 softmax + 温度 T」で勝率にしただけだった（T=1 は較正が合わず、T を検証 fold で最適化すると二値・softmax と同程度、ただし楽観値）。
ここでは、**より柔軟な確率化**を試し、**アンサンブル v2 の候補として有意に効くか**を確かめる。順位を直接学習するモデルは、二値分類とは違う誤差を出すので、較正できれば合成で効く可能性がある。

## 確率化の方式（すべて他の年だけで当てはめる = leave-one-year-out）
| 方式 | 内容 |
|---|---|
| T | レース内 softmax(score / T)。T は他の年で最適化 |
| Platt | 馬のスコアをレース平均で引いた値に対するロジスティック回帰（p_win・p_top2・p_top3 を別々）→ レース内で合計を 1・2・3 にそろえる |
| 特徴 LR | スコアのレース内 z・順位・最大との差・頭数を入力にしたロジスティック回帰（別々）→ 合計をそろえる |
| 等張 | レース内 softmax(score) の確率を等張回帰で較正 → 合計をそろえる |

特徴量・パラメータは v2 の LightGBM と同じ（追い切り・厩舎コメント・血統×条件を含む）。採用基準 = LogLoss の差の 95% 信頼区間の上限 < 0 かつ連対・複勝が悪化しない（0.0002 以内）。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, win_models as wm, zoo, ensemble as en
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_l"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP05L_THREADS", detect_compute(reserve_cores=6).threads))
mlflow.set_experiment("keiba-tp-segment")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
key = ["race_id", "horse_id"]
race_all, fin_all, year_all = meta["race_id"].to_numpy(), meta["finish_position"].to_numpy(), meta["year"].to_numpy()

In [ ]:
# ランキング系のスコアの OOF（各 fold は過去年で学習 → 当該年のスコア）
SC = {}
for kind in ("lambdarank", "xendcg"):
    t0 = time.time()
    s = np.full(len(meta), np.nan)
    for f in folds:
        tr, va = f.train_index, f.valid_index
        Y = {"win": (fin_all[tr] == 1).astype(int), "top2": (fin_all[tr] <= 2).astype(int), "top3": (fin_all[tr] <= 3).astype(int), "finish": fin_all[tr]}
        o = wm.fit_predict(kind, X.iloc[tr], Y, race_all[tr], X.iloc[va], race_all[va], THREADS, seed=0, params=LP["lgb"], n_rounds=LP["n_rounds"])
        s[va] = o["win_score"]
    SC[kind] = s
    print(kind, f"{time.time() - t0:.0f}s", flush=True)
    gc.collect()
idx = np.flatnonzero(~np.isnan(SC["lambdarank"]))
S = meta.iloc[idx][key + ["year", "finish_position"]].reset_index(drop=True)
race, years, fin = S["race_id"].to_numpy(), S["year"].to_numpy(), S["finish_position"].to_numpy()
fsize = S.groupby("race_id")["race_id"].transform("size").to_numpy()
Yt = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
ref = pd.read_parquet(OOF_DIR / "lgb_extras_ped.parquet").merge(S[key], on=key, how="right")
v2 = pd.read_parquet(OOF_DIR / "ensemble_v2.parquet").merge(S[key], on=key, how="right")
print("参照 lgb_extras_ped win LL", round(wm.binary_logloss(ref["p_win"].to_numpy(), Yt["win"]), 5), "/ v2 win LL", round(wm.binary_logloss(v2["p_win"].to_numpy(), Yt["win"]), 5))

In [ ]:
def race_feats(sc):
    g = pd.DataFrame({"r": race, "s": sc})
    mu, sd = g.groupby("r")["s"].transform("mean"), g.groupby("r")["s"].transform("std").replace(0, np.nan).fillna(1.0)
    rk = g.groupby("r")["s"].rank(ascending=False, method="average")
    mx = g.groupby("r")["s"].transform("max")
    return np.column_stack([sc - mu, (sc - mu) / sd, rk, sc - mx, np.log(fsize)])


def convert(sc, how):
    """スコア → {win, top2, top3}（LOYO。他の年で当てはめ、レース内の合計をそろえる）。"""
    out = {t: np.zeros(len(sc)) for t in TOT}
    F = race_feats(sc)
    for y in np.unique(years):
        tr, te = years != y, years == y
        if how == "T":
            def nll(T):
                p = wm.softmax_by_race(race[tr], sc[tr], T)
                return wm.binary_logloss(p, Yt["win"][tr])
            T = minimize_scalar(nll, bounds=(0.2, 5.0), method="bounded").x
            pw = wm.softmax_by_race(race[te], sc[te], T)
            out["win"][te] = pw
            out["top2"][te], out["top3"][te] = zoo.harville_topk(race[te], pw, 0.8, 0.7)
        elif how == "Platt":
            for t in TOT:
                lr = LogisticRegression(C=1e4, max_iter=300).fit(F[tr][:, :1], Yt[t][tr])
                out[t][te] = lr.predict_proba(F[te][:, :1])[:, 1]
        elif how == "特徴LR":
            for t in TOT:
                lr = LogisticRegression(C=1.0, max_iter=500).fit(F[tr], Yt[t][tr])
                out[t][te] = lr.predict_proba(F[te])[:, 1]
        elif how == "等張":
            p_tr = wm.softmax_by_race(race[tr], sc[tr], 1.0); p_te = wm.softmax_by_race(race[te], sc[te], 1.0)
            for t in TOT:
                iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6).fit(p_tr, Yt[t][tr])
                base = iso.predict(p_te) if t == "win" else None
                if t == "win":
                    out["win"][te] = base
                else:
                    pw = out["win"][te]
                    p2, p3 = zoo.harville_topk(race[te], pw, 0.8, 0.7)
                    out["top2"][te], out["top3"][te] = p2, p3
    return {t: wm.normalize_to_total(race, np.clip(out[t], 1e-6, 1 - 1e-6), TOT[t]) for t in TOT}


def to_oof(P):
    o = S[key + ["year"]].copy()
    for t in TOT:
        o[f"p_{t}"] = P[t]
    return o


def cmp(P, refdf, name):
    out = {"variant": name}
    for t in TOT:
        b = wm.paired_bootstrap_diff(wm.per_race_loss(race, refdf[f"p_{t}"].to_numpy(), Yt[t]), wm.per_race_loss(race, P[t], Yt[t]), n_boot=500)
        out[f"{t}_ref"], out[f"{t}_new"], out[f"Δ{t}"], out[f"{t}_hi"] = wm.binary_logloss(refdf[f"p_{t}"].to_numpy(), Yt[t]), wm.binary_logloss(P[t], Yt[t]), b["diff"], b["hi"]
    return out


CONV, ROWS = {}, []
for kind in SC:
    for how in ("T", "Platt", "特徴LR", "等張"):
        P = convert(SC[kind], how)
        CONV[(kind, how)] = P
        ROWS.append(cmp(P, ref, f"{kind} / {how}（vs lgb_extras_ped）"))
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_ref", "win_new", "Δwin", "Δtop2", "Δtop3"]].round(5)); R.to_csv(OUT / "conversion_vs_lgb.csv")

## アンサンブル v2 の候補に加える（LOYO）
各方式の中で単独の勝率 LogLoss が最良のものを、lambdarank・rank_xendcg それぞれ 1 つずつ選んで v2 の 7 モデルに加え、v2 のアンサンブルと paired bootstrap で比べる。

In [ ]:
pick = {}
for kind in SC:
    cand = {how: wm.binary_logloss(CONV[(kind, how)]["win"], Yt["win"]) for how in ("T", "Platt", "特徴LR", "等張")}
    pick[kind] = min(cand, key=cand.get)
    to_oof(CONV[(kind, pick[kind])]).to_parquet(OOF_DIR / f"{kind}_calibrated.parquet", index=False)
print("各ランキング系で単独最良の確率化:", pick)
v2s = json.loads((WORK / "ensemble_v2/ensemble_spec.json").read_text(encoding="utf-8"))
names = v2s["models"] + [f"{k}_calibrated" for k in SC]
base = S[key + ["year", "finish_position"]].copy()
for n in names:
    o = pd.read_parquet(OOF_DIR / f"{n}.parquet")[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in TOT})
    base = base.merge(o, on=key, how="inner")
assert len(base) == len(S)
P3 = {t: np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1) for t in TOT}
ORDER = ["mean", "logit_mean", "greedy", "weighted_logit", "stack_lr"]
rows, FIN = [], {}
for t in TOT:
    sub, outs = {}, {}
    for m_ in en.METHODS:
        v, out = en.loyo_eval(m_, P3[t], years, race, Yt[t], TOT[t])
        sub[m_], outs[m_] = v, out
    ch = next(m_ for m_ in ORDER if sub[m_] <= min(sub[x] for x in ORDER) + 5e-5)
    FIN[t] = outs[ch]
    b = wm.paired_bootstrap_diff(wm.per_race_loss(race, v2[f"p_{t}"].to_numpy(), Yt[t]), wm.per_race_loss(race, FIN[t], Yt[t]), n_boot=1000)
    rows.append({"target": t, "v2": en.ll(v2[f"p_{t}"].to_numpy(), Yt[t]), "v2+ランキング系": en.ll(FIN[t], Yt[t]), "合成案": ch, "Δ": b["diff"], "hi": b["hi"], "採用": b["hi"] < 0})
ENS = pd.DataFrame(rows).set_index("target")
display(ENS.round(5)); ENS.to_csv(OUT / "ensemble_with_ranking.csv")
ADOPT = bool(ENS.loc["win", "採用"] and ENS.loc["top2", "Δ"] <= 2e-4 and ENS.loc["top3", "Δ"] <= 2e-4)
print("v2 にランキング系を加える案を採用:", ADOPT)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "conversion": R.round(6).reset_index().to_dict("records"), "picked": pick, "ensemble": ENS.round(6).reset_index().to_dict("records"), "adopt": ADOPT}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-05l-ranking-calibration", tags={"notebook": "tp-05l", "git_commit": GIT_COMMIT}):
    mlflow.log_metrics({f"ens_delta_{t}": float(ENS.loc[t, "Δ"]) for t in TOT})
    for f_ in ("conversion_vs_lgb.csv", "ensemble_with_ranking.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_l")